In [ ]:
# -*- coding: utf-8 -*-
"""
Final XGBoost scenario analysis workflow.

This script:
1. Reads the predefined training and test datasets directly.
2. Fits the final XGBoost model without re-splitting the data.
3. Enumerates all 1,440 operational combinations using the training backgrounds.
4. Extracts the Top 50 and Top 100 high-score combinations.
5. Calculates category frequencies and marginal predicted scores.
6. Automatically identifies the stable high-frequency operational categories.
7. Constructs S0-S4 management scenarios using the fixed test backgrounds.
8. Calculates paired bootstrap confidence intervals for scenario improvements.
9. Exports all retained scenario-analysis results to one English Excel workbook.
"""

from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

In [ ]:
# ============================================================
# 1. File paths
# ============================================================

TRAIN_FILE = Path(r"D:\training_set.xlsx")
TEST_FILE = Path(r"D:\test_set.xlsx")

OUTPUT_FILE = Path(r"D:\scenario_analysis_results.xlsx")

In [ ]:
# ============================================================
# 2. Input features and target
# ============================================================

NUMERIC_FEATURES = [
    "Bulk density",
    "Initial C/N ratio",
    "Initial moisture content",
    "Initial pH",
    "Composting duration",
    "Reactor volume",
]

CATEGORICAL_FEATURES = [
    "Composting type",
    "Bulking agent type",
    "Treatment condition",
    "Turning regime",
    "Aeration regime",
]

TARGET = "Score"

In [ ]:
# ============================================================
# 3. Fixed category orders
# ============================================================

# These explicit category orders preserve the encoded feature order
# used by the original finalized model before the dataset was translated
# into English. This prevents small XGBoost differences caused only by
# changes in alphabetical category ordering after translation.

CATEGORY_ORDERS = {
    "Composting type": [
        "Single-material composting",
        "Mixed-material composting",
    ],

    "Bulking agent type": [
        "No bulking agent",
        "Other",
        "Mixed bulking agents",
        "Wood chips",
        "Leaves",
        "Crop stalk",
        "Rice husk",
        "Wheat bran",
    ],

    "Treatment condition": [
        "No adjustment",
        "Combined treatment",
        "Microbial inoculant",
        "Physicochemical additive",
        "Adjustment of initial C/N or moisture content",
        "Pretreatment",
    ],

    "Turning regime": [
        "No turning",
        "Variable turning",
        "Fixed turning",
    ],

    "Aeration regime": [
        "No aeration",
        "Variable continuous aeration",
        "Variable intermittent aeration",
        "Fixed continuous aeration",
        "Fixed intermittent aeration",
    ],
}

In [ ]:
# ============================================================
# 4. Read the predefined training and test datasets
# ============================================================

train_df = pd.read_excel(TRAIN_FILE)
test_df = pd.read_excel(TEST_FILE)

required_columns = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
    + [TARGET]
)

missing_train_columns = [
    column
    for column in required_columns
    if column not in train_df.columns
]

missing_test_columns = [
    column
    for column in required_columns
    if column not in test_df.columns
]

if missing_train_columns:
    raise ValueError(
        "Training dataset is missing the following columns: "
        f"{missing_train_columns}"
    )

if missing_test_columns:
    raise ValueError(
        "Test dataset is missing the following columns: "
        f"{missing_test_columns}"
    )

if train_df[required_columns].isna().sum().sum() > 0:
    raise ValueError(
        "Missing values were detected in the training dataset."
    )

if test_df[required_columns].isna().sum().sum() > 0:
    raise ValueError(
        "Missing values were detected in the test dataset."
    )

In [ ]:
# ============================================================
# 5. Validate categorical values
# ============================================================

for feature in CATEGORICAL_FEATURES:

    allowed_categories = set(CATEGORY_ORDERS[feature])

    train_categories = set(
        train_df[feature].dropna().unique()
    )

    test_categories = set(
        test_df[feature].dropna().unique()
    )

    unknown_train = (
        train_categories - allowed_categories
    )

    unknown_test = (
        test_categories - allowed_categories
    )

    if unknown_train:
        raise ValueError(
            f"Unexpected categories in training feature "
            f"'{feature}': {unknown_train}"
        )

    if unknown_test:
        raise ValueError(
            f"Unexpected categories in test feature "
            f"'{feature}': {unknown_test}"
        )


print("Training samples:", len(train_df))
print("Test samples:    ", len(test_df))

In [ ]:
# ============================================================
# 6. Build model inputs
# ============================================================

X_train = train_df[
    NUMERIC_FEATURES + CATEGORICAL_FEATURES
].copy()

y_train = (
    train_df[TARGET]
    .astype(float)
    .to_numpy()
)

X_test = test_df[
    NUMERIC_FEATURES + CATEGORICAL_FEATURES
].copy()

y_test = (
    test_df[TARGET]
    .astype(float)
    .to_numpy()
)

In [ ]:
# ============================================================
# 7. Data preprocessing
# ============================================================

one_hot_categories = [
    CATEGORY_ORDERS[feature]
    for feature in CATEGORICAL_FEATURES
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            NUMERIC_FEATURES,
        ),
        (
            "cat",
            OneHotEncoder(
                categories=one_hot_categories,
                handle_unknown="ignore",
                sparse_output=False,
            ),
            CATEGORICAL_FEATURES,
        ),
    ]
)

In [ ]:
# ============================================================
# 8. Final XGBoost model
# ============================================================

xgb_model = XGBRegressor(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.07,
    subsample=0.90,
    colsample_bytree=1.00,
    min_child_weight=3,
    gamma=0,
    reg_alpha=0,
    reg_lambda=3,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
)


model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("xgboost", xgb_model),
    ]
)

In [ ]:
# ============================================================
# 9. Fit the final model
# ============================================================

model.fit(
    X_train,
    y_train,
)

print("Final XGBoost model fitted.")

In [ ]:
# ============================================================
# 10. Enumerate all 1,440 operational combinations
# ============================================================

COMPOSTING_TYPES = (
    CATEGORY_ORDERS["Composting type"]
)

BULKING_AGENTS = (
    CATEGORY_ORDERS["Bulking agent type"]
)

TREATMENT_CONDITIONS = (
    CATEGORY_ORDERS["Treatment condition"]
)

TURNING_REGIMES = (
    CATEGORY_ORDERS["Turning regime"]
)

AERATION_REGIMES = (
    CATEGORY_ORDERS["Aeration regime"]
)


all_combination_results = []


for (
    composting_type,
    bulking_agent,
    treatment_condition,
    turning_regime,
    aeration_regime,
) in product(
    COMPOSTING_TYPES,
    BULKING_AGENTS,
    TREATMENT_CONDITIONS,
    TURNING_REGIMES,
    AERATION_REGIMES,
):

    scenario_data = X_train.copy()

    scenario_data[
        "Composting type"
    ] = composting_type

    scenario_data[
        "Bulking agent type"
    ] = bulking_agent

    scenario_data[
        "Treatment condition"
    ] = treatment_condition

    scenario_data[
        "Turning regime"
    ] = turning_regime

    scenario_data[
        "Aeration regime"
    ] = aeration_regime


    predicted_score = model.predict(
        scenario_data
    )


    all_combination_results.append(
        {
            "Composting type":
                composting_type,

            "Bulking agent type":
                bulking_agent,

            "Treatment condition":
                treatment_condition,

            "Turning regime":
                turning_regime,

            "Aeration regime":
                aeration_regime,

            "Mean predicted Score":
                float(
                    np.mean(
                        predicted_score
                    )
                ),

            "SD":
                float(
                    np.std(
                        predicted_score,
                        ddof=1,
                    )
                ),

            "Min":
                float(
                    np.min(
                        predicted_score
                    )
                ),

            "Max":
                float(
                    np.max(
                        predicted_score
                    )
                ),
        }
    )


all_combinations_df = pd.DataFrame(
    all_combination_results
)


all_combinations_df = (
    all_combinations_df
    .sort_values(
        by="Mean predicted Score",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)


all_combinations_df.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(all_combinations_df) + 1,
    ),
)


print(
    "Total operational combinations:",
    len(all_combinations_df),
)

In [ ]:
# ============================================================
# 11. Extract Top 50 and Top 100 combinations
# ============================================================

top50_df = (
    all_combinations_df
    .head(50)
    .copy()
)

top100_df = (
    all_combinations_df
    .head(100)
    .copy()
)

In [ ]:
# ============================================================
# 12. Calculate marginal predicted scores for each category
# ============================================================

marginal_results = []


for feature in CATEGORICAL_FEATURES:

    for category in CATEGORY_ORDERS[feature]:

        category_scores = (
            all_combinations_df.loc[
                all_combinations_df[
                    feature
                ] == category,
                "Mean predicted Score",
            ]
        )


        marginal_results.append(
            {
                "Variable":
                    feature,

                "Operational category":
                    category,

                "Mean score":
                    category_scores.mean(),

                "SD score":
                    category_scores.std(
                        ddof=1
                    ),

                "Median score":
                    category_scores.median(),

                "Min score":
                    category_scores.min(),

                "Max score":
                    category_scores.max(),
            }
        )


marginal_score_df = pd.DataFrame(
    marginal_results
)


marginal_score_df = (
    marginal_score_df
    .sort_values(
        by="Mean score",
        ascending=False,
    )
    .reset_index(drop=True)
)

In [ ]:
# ============================================================
# 13. Top 50 and Top 100 category-frequency analysis
# ============================================================

frequency_rows = []

selected_stable_categories = {}


for feature in CATEGORICAL_FEATURES:

    feature_rows = []

    for category in CATEGORY_ORDERS[feature]:

        top50_count = int(
            (
                top50_df[
                    feature
                ] == category
            ).sum()
        )

        top100_count = int(
            (
                top100_df[
                    feature
                ] == category
            ).sum()
        )


        top50_frequency = (
            top50_count / 50
        )

        top100_frequency = (
            top100_count / 100
        )

        mean_frequency = (
            top50_frequency
            + top100_frequency
        ) / 2


        marginal_mean_score = (
            all_combinations_df.loc[
                all_combinations_df[
                    feature
                ] == category,
                "Mean predicted Score",
            ]
            .mean()
        )


        feature_rows.append(
            {
                "Variable":
                    feature,

                "Category":
                    category,

                "Top50 count":
                    top50_count,

                "Top50 frequency":
                    top50_frequency,

                "Top100 count":
                    top100_count,

                "Top100 frequency":
                    top100_frequency,

                "Mean Top50/100 frequency":
                    mean_frequency,

                "1440-combination marginal mean Score":
                    marginal_mean_score,
            }
        )


    feature_frequency_df = pd.DataFrame(
        feature_rows
    )


    feature_frequency_df = (
        feature_frequency_df
        .sort_values(
            by=[
                "Mean Top50/100 frequency",
                "1440-combination marginal mean Score",
            ],
            ascending=[
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )


    stable_category = (
        feature_frequency_df
        .iloc[0]["Category"]
    )


    selected_stable_categories[
        feature
    ] = stable_category


    for _, row in (
        feature_frequency_df
        .iterrows()
    ):

        is_stable = (
            row["Category"]
            == stable_category
        )


        if (
            feature
            == "Composting type"
            and is_stable
        ):

            scenario_use = (
                "Interpretation only; "
                "S2-S4 retain the original "
                "composting type"
            )

        elif (
            feature
            != "Composting type"
            and is_stable
        ):

            scenario_use = (
                "Used in S2-S4"
            )

        else:

            scenario_use = (
                "Not used"
            )


        frequency_rows.append(
            {
                "Variable":
                    row[
                        "Variable"
                    ],

                "Category":
                    row[
                        "Category"
                    ],

                "Top50 count":
                    row[
                        "Top50 count"
                    ],

                "Top50 frequency":
                    row[
                        "Top50 frequency"
                    ],

                "Top100 count":
                    row[
                        "Top100 count"
                    ],

                "Top100 frequency":
                    row[
                        "Top100 frequency"
                    ],

                "Mean Top50/100 frequency":
                    row[
                        "Mean Top50/100 frequency"
                    ],

                "1440-combination marginal mean Score":
                    row[
                        "1440-combination marginal mean Score"
                    ],

                "Stable high-frequency category":
                    (
                        "Yes"
                        if is_stable
                        else "No"
                    ),

                "Scenario use":
                    scenario_use,
            }
        )


frequency_df = pd.DataFrame(
    frequency_rows
)

In [ ]:
# ============================================================
# 14. Operational settings used in S2-S4
# ============================================================

scenario_operation_settings = {
    "Bulking agent type":
        selected_stable_categories[
            "Bulking agent type"
        ],

    "Treatment condition":
        selected_stable_categories[
            "Treatment condition"
        ],

    "Turning regime":
        selected_stable_categories[
            "Turning regime"
        ],

    "Aeration regime":
        selected_stable_categories[
            "Aeration regime"
        ],
}


print(
    "\nStable operational settings "
    "used in S2-S4:"
)

for feature, category in (
    scenario_operation_settings
    .items()
):

    print(
        f"{feature}: {category}"
    )

In [ ]:
# ============================================================
# 15. Construct S0
# ============================================================

S0 = X_test.copy()

In [ ]:
# ============================================================
# 16. Construct S1
# ============================================================

# S1 applies unified numerical conditions.
# Reactor volume and all categorical variables remain unchanged.

S1 = X_test.copy()

S1[
    "Bulk density"
] = 0.15

S1[
    "Initial C/N ratio"
] = 30.0

S1[
    "Initial moisture content"
] = 65.0

S1[
    "Initial pH"
] = 4.5

S1[
    "Composting duration"
] = 40.0

In [ ]:
# ============================================================
# 17. Construct S2
# ============================================================

# S2 retains all original numerical conditions and the original
# composting type, while applying the stable operational settings.

S2 = X_test.copy()

for (
    feature,
    category,
) in scenario_operation_settings.items():

    S2[
        feature
    ] = category

In [ ]:
# ============================================================
# 18. Construct S3
# ============================================================

# S3 combines the unified numerical settings from S1
# with the stable operational settings from S2.

S3 = S1.copy()

for (
    feature,
    category,
) in scenario_operation_settings.items():

    S3[
        feature
    ] = category

In [ ]:
# ============================================================
# 19. Construct S4
# ============================================================

# S4 only corrects numerical values outside the recommended ranges.
# Values already inside the recommended ranges remain unchanged.
# Reactor volume and composting type remain unchanged.

S4 = X_test.copy()


S4[
    "Bulk density"
] = (
    S4[
        "Bulk density"
    ]
    .clip(
        lower=0.10,
        upper=0.20,
    )
)


S4[
    "Initial C/N ratio"
] = (
    S4[
        "Initial C/N ratio"
    ]
    .clip(
        lower=25.0,
        upper=30.0,
    )
)


S4[
    "Initial moisture content"
] = (
    S4[
        "Initial moisture content"
    ]
    .clip(
        lower=60.0,
        upper=65.0,
    )
)


S4[
    "Initial pH"
] = np.where(
    S4[
        "Initial pH"
    ] < 4.5,
    4.5,
    S4[
        "Initial pH"
    ],
)


S4[
    "Composting duration"
] = (
    S4[
        "Composting duration"
    ]
    .clip(
        lower=40.0,
        upper=60.0,
    )
)


for (
    feature,
    category,
) in scenario_operation_settings.items():

    S4[
        feature
    ] = category

In [ ]:
# ============================================================
# 20. Predict S0-S4
# ============================================================

scenario_data = {
    "S0": S0,
    "S1": S1,
    "S2": S2,
    "S3": S3,
    "S4": S4,
}


scenario_predictions = {
    scenario:
        model.predict(data)

    for (
        scenario,
        data,
    ) in scenario_data.items()
}


baseline_prediction = (
    scenario_predictions["S0"]
)

In [ ]:
# ============================================================
# 21. Paired bootstrap confidence intervals
# ============================================================

BOOTSTRAP_REPETITIONS = 5000
BOOTSTRAP_RANDOM_STATE = 42

bootstrap_rng = np.random.default_rng(
    BOOTSTRAP_RANDOM_STATE
)


def paired_bootstrap_ci(
    score_difference,
    rng,
    repetitions=5000,
):

    score_difference = np.asarray(
        score_difference,
        dtype=float,
    )

    n_samples = len(
        score_difference
    )


    bootstrap_indices = (
        rng.integers(
            low=0,
            high=n_samples,
            size=(
                repetitions,
                n_samples,
            ),
        )
    )


    bootstrap_means = (
        score_difference[
            bootstrap_indices
        ]
        .mean(axis=1)
    )


    ci_low = np.percentile(
        bootstrap_means,
        2.5,
    )

    ci_high = np.percentile(
        bootstrap_means,
        97.5,
    )


    return (
        float(ci_low),
        float(ci_high),
    )

In [ ]:
# ============================================================
# 22. Build S0-S4 scenario summary
# ============================================================

scenario_summary_rows = []


baseline_mean = float(
    np.mean(
        baseline_prediction
    )
)


for scenario in [
    "S0",
    "S1",
    "S2",
    "S3",
    "S4",
]:

    prediction = (
        scenario_predictions[
            scenario
        ]
    )


    score_difference = (
        prediction
        - baseline_prediction
    )


    mean_score = float(
        np.mean(
            prediction
        )
    )

    sd_score = float(
        np.std(
            prediction,
            ddof=1,
        )
    )

    median_score = float(
        np.median(
            prediction
        )
    )

    min_score = float(
        np.min(
            prediction
        )
    )

    max_score = float(
        np.max(
            prediction
        )
    )


    if scenario == "S0":

        mean_change = 0.0
        relative_change = np.nan
        ci_low = np.nan
        ci_high = np.nan
        improved_n = np.nan
        improved_proportion = np.nan

    else:

        mean_change = float(
            np.mean(
                score_difference
            )
        )

        relative_change = (
            mean_change
            / baseline_mean
        )


        (
            ci_low,
            ci_high,
        ) = paired_bootstrap_ci(
            score_difference,
            bootstrap_rng,
            repetitions=
                BOOTSTRAP_REPETITIONS,
        )


        improved_n = int(
            np.sum(
                score_difference > 0
            )
        )


        improved_proportion = (
            improved_n
            / len(
                score_difference
            )
        )


    scenario_summary_rows.append(
        {
            "Scenario":
                scenario,

            "Mean predicted Score":
                mean_score,

            "SD":
                sd_score,

            "Median":
                median_score,

            "Min":
                min_score,

            "Max":
                max_score,

            "Mean Score change vs S0":
                mean_change,

            "Relative change":
                relative_change,

            "95% CI low":
                ci_low,

            "95% CI high":
                ci_high,

            "Improved n":
                improved_n,

            "Improved proportion":
                improved_proportion,
        }
    )


scenario_summary_df = pd.DataFrame(
    scenario_summary_rows
)

In [ ]:
# ============================================================
# 23. Export all retained results to Excel
# ============================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl",
) as writer:

    scenario_summary_df.to_excel(
        writer,
        sheet_name=
            "01_Scenario_Summary",
        index=False,
        startrow=1,
    )


    all_combinations_df.to_excel(
        writer,
        sheet_name=
            "09_All_1440_Combinations",
        index=False,
    )


    top50_df.to_excel(
        writer,
        sheet_name=
            "02_Top50_Combinations",
        index=False,
    )


    top100_df.to_excel(
        writer,
        sheet_name=
            "03_Top100_Combinations",
        index=False,
    )


    frequency_df.to_excel(
        writer,
        sheet_name=
            "04_Top50_100_Frequency",
        index=False,
        startrow=1,
    )


    marginal_score_df.to_excel(
        writer,
        sheet_name=
            "05_Category_Marginal_Scores",
        index=False,
    )


    workbook = writer.book


    scenario_sheet = workbook[
        "01_Scenario_Summary"
    ]

    scenario_sheet[
        "A1"
    ] = (
        "S0-S4 theoretical prediction "
        "summary using the fixed test set"
    )


    frequency_sheet = workbook[
        "04_Top50_100_Frequency"
    ]

    frequency_sheet[
        "A1"
    ] = (
        "Category frequencies in the "
        "Top 50 and Top 100 "
        "high-score operational combinations"
    )


    for sheet in workbook.worksheets:

        sheet.freeze_panes = (
            "A3"
            if sheet.title
            in [
                "01_Scenario_Summary",
                "04_Top50_100_Frequency",
            ]
            else "A2"
        )


        for column_cells in (
            sheet.columns
        ):

            maximum_length = 0

            column_letter = (
                column_cells[0]
                .column_letter
            )

            for cell in column_cells:

                if cell.value is not None:

                    cell_length = len(
                        str(
                            cell.value
                        )
                    )

                    maximum_length = max(
                        maximum_length,
                        cell_length,
                    )


            sheet.column_dimensions[
                column_letter
            ].width = min(
                maximum_length + 2,
                45,
            )

In [ ]:
# ============================================================
# 24. Final information
# ============================================================

print(
    "\nScenario analysis completed."
)

print(
    "Output file:",
    OUTPUT_FILE,
)

print(
    "\nScenario summary:"
)

print(
    scenario_summary_df
)